# Data Engineering: Where Does LLM Training Data Come From, and How Do We Clean It?

> **Background**: LLaMA 3 reportedly trained on ~15 trillion tokens. Where do 15T tokens come from? They do not fall from the sky.
>
> Goal for this part: **understand the full pipeline from raw internet HTML to clean training text: what each step does, and why it exists.**

Core metaphor: **data engineering is cooking for an LLM.**
- the internet is a huge market (everything is there, but it is dirty)
- data engineering is washing, trimming, cutting, and mixing ingredients
- garbage in, garbage out

We will use concrete text snippets to make every step tangible.


In [1]:
import re
import hashlib
import numpy as np

np.random.seed(42)

### 0. Bird's-eye view: the data pipeline

Start with the full picture, then break it down step by step:

```
  Common Crawl (raw HTML, ~500TB/month)
       |
       v
  +----------------+
  | 1. Text extract |  HTML -> text, remove nav/ads/scripts
  +--------+-------+
           v
  +----------------+
  | 2. Lang filter  |  keep only target languages
  +--------+-------+
           v
  +----------------+
  | 3. Quality filt |  remove junk/garbled/too short/too long
  +--------+-------+
           v
  +----------------+
  | 4. Deduplicate  |  exact dedup + MinHash near-dup
  +--------+-------+
           v
  +----------------+
  | 5. Data mixing  |  CC + Wiki + Books + Code
  +--------+-------+
           v
      Clean training data
```


---

### 1. Text extraction: mining gold from an HTML garbage heap

#### 1.1 What is inside Common Crawl?

Common Crawl is a non-profit that crawls a large slice of the web every month.
But raw HTML contains a lot of non-content:

- navigation bars: "Home | About | Contact"
- pop-up ads: "Limited-time offer! Buy now!"
- JavaScript: `function trackUser(){...}`
- CSS: `.sidebar { float: right; }`
- footers: "Copyright 2024. All rights reserved."
- comment sections: "First!"

A normal article is buried under all that, like picking vegetables out of plastic bags, dirt, and rotten leaves.

#### 1.2 Demonstrate with a toy HTML


In [2]:
# === : Common Crawl HTML ===
raw_html = """
<!DOCTYPE html>
<html>
<head>
  <title>What is Machine Learning? - AI Blog</title>
  <meta name="description" content="Learn ML basics">
  <script src="tracking.js"></script>
  <style>.ad { color: red; }</style>
</head>
<body>
  <nav>
    <a href="/">Home</a> |
    <a href="/about">About</a> |
    <a href="/contact">Contact</a>
  </nav>
  
  <div class="sidebar">
    <div class="ad">
      <h3>Sponsored</h3>
      <p>Buy the best AI course! 50% off today only!</p>
      <button>Click Here!</button>
    </div>
    <script type="text/javascript">
      var _gaq = _gaq || [];
      _gaq.push(['_setAccount', 'UA-12345-6']);
      _gaq.push(['_trackPageview']);
    </script>
  </div>
  
  <article>
    <h1>What is Machine Learning?</h1>
    <p>Machine learning is a subset of artificial intelligence
    that enables systems to learn and improve from experience
    without being explicitly programmed.</p>
    
    <p>The process of learning begins with observations or data,
    such as examples, direct experience, or instruction.</p>
    
    <p>Machine learning algorithms build a mathematical model
    based on sample data, known as "training data".</p>
  </article>
  
  <div class="comments">
    <p>User123: Great article!</p>
    <p>Bot456: Buy followers at cheap-followers.com!</p>
  </div>
  
  <footer>
    <p>Copyright 2024 AI Blog. All rights reserved.</p>
    <p>Terms of Service | Privacy Policy | Cookie Settings</p>
  </footer>
</body>
</html>
"""

print("=== Raw HTML ===")
print(raw_html[:500])
print("...")
print()
print("↑ ： 、 、JS 、 、 ")

=== Raw HTML ===

<!DOCTYPE html>
<html>
<head>
  <title>What is Machine Learning? - AI Blog</title>
  <meta name="description" content="Learn ML basics">
  <script src="tracking.js"></script>
  <style>.ad { color: red; }</style>
</head>
<body>
  <nav>
    <a href="/">Home</a> |
    <a href="/about">About</a> |
    <a href="/contact">Contact</a>
  </nav>

  <div class="sidebar">
    <div class="ad">
      <h3>Sponsored</h3>
      <p>Buy the best AI course! 50% off today only!</p>
      <button>Click Here!</butto
...

↑ ： 、 、JS 、 、 


In [3]:
# === Manual demo:Text extraction ===
print("=== Text extraction：HTML -> ===")
print()

# Step 1: Remove <script> <style> tags 
def remove_scripts_styles(html):
    """Remove script style tags( )"""
    html = re.sub(r'<script[^>]*>.*?</script>', '', html, flags=re.DOTALL | re.IGNORECASE)
    html = re.sub(r'<style[^>]*>.*?</style>', '', html, flags=re.DOTALL | re.IGNORECASE)
    return html

# Step 2: Remove HTML tags
def strip_tags(html):
    """Remove <...> tags"""
    return re.sub(r'<[^>]+>', '', html)

# Step 3: Cleanwhitespace
def clean_whitespace(text):
    """ 、 whitespace"""
    text = re.sub(r'[ \t]+', ' ', text)  #  /tab
    text = re.sub(r'\n\s*\n\s*\n+', '\n\n', text)  #  -> 
    return text.strip()

#  
print("Step 1 — Remove script/style tags:")
no_scripts = remove_scripts_styles(raw_html)
print(f" raw: {len(raw_html)} chars -> removeafter: {len(no_scripts)} chars")
print()

print("Step 2 — Remove HTML tags:")
plain_text = strip_tags(no_scripts)
print(f" removetagsafter: {len(plain_text)} chars")
print()

print("Step 3 — Cleanwhitespace:")
clean_text = clean_whitespace(plain_text)
print(f" normalize whitespaceafter: {len(clean_text)} chars")
print()

print("=" * 60)
print(" :")
print("=" * 60)
print(clean_text)
print("=" * 60)
print()
print(" ： (Home|About|Contact) 、 、 ")
print(" -> 「quality 」 ")

=== Text extraction：HTML -> ===

Step 1 — Remove script/style tags:
 raw: 1454 chars -> removeafter: 1226 chars

Step 2 — Remove HTML tags:
 removetagsafter: 831 chars

Step 3 — Cleanwhitespace:
 normalize whitespaceafter: 707 chars

 :
What is Machine Learning? - AI Blog

 Home |
 About |
 Contact

 Sponsored
 Buy the best AI course! 50% off today only!
 Click Here!

 What is Machine Learning?
 Machine learning is a subset of artificial intelligence
 that enables systems to learn and improve from experience
 without being explicitly programmed.

 The process of learning begins with observations or data,
 such as examples, direct experience, or instruction.

 Machine learning algorithms build a mathematical model
 based on sample data, known as "training data".

 User123: Great article!
 Bot456: Buy followers at cheap-followers.com!

 Copyright 2024 AI Blog. All rights reserved.
 Terms of Service | Privacy Policy | Cookie Settings

 ： (Home|About|Contact) 、 、 
 -> 「quality 」 


---

### 2. Quality filtering: is this document worth learning from?

After extracting plain text, you still need to remove junk.
A common approach is a multi-layer filter.

#### 2.1 Heuristic rules (cheap and fast)

Like checking vegetables:
- too short -> likely templated junk
- too long -> possibly concatenated spam
- too many symbols -> garbled text
- too many repeated lines -> boilerplate pages


In [4]:
# === quality rule： ===
print("=== quality rule ===")
print()

#  「 」 
samples = [
    {"name": " document", "text": "Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed. The field has grown rapidly since the 2010s, driven by advances in deep learning and the availability of large datasets."},
    {"name": " ", "text": "BUY NOW!!! Click here!!! Limited time offer!!! Subscribe today and get 50% OFF!!! Don't miss this opportunity!!!"},
    {"name": " ", "text": "Chapter 1. Introduction. Chapter 2. Methods. Chapter 3. Results. Chapter 4. Discussion. Chapter 5. Conclusion. Appendix A. Appendix B. References."},
    {"name": "too short", "text": "Hello world."},
    {"name": "random chars", "text": "asdfjkl; qwerty zxcvbnm @#$%@# 123456789 !!!!!!!"},
    {"name": "duplicate ", "text": "This is a blog post.\n" * 30 + "unique content here"},
]

def quality_check(text):
    """ ( keep, drop )"""
    
    # rule 1: 
    words = text.split()
    if len(words) < 5:
        return False, f"too short ({len(words)} words < 5)"
    if len(text) > 5000:
        return False, f"too long ({len(text)} chars > 5000)"
    
    # rule 2: — 3-10, 
    avg_word_len = np.mean([len(w) for w in words])
    if avg_word_len > 12:
        return False, f"avg word length abnormal ({avg_word_len:.1f} > 12)"
    
    # rule 3: share — document 15%
    special_count = sum(1 for c in text if not c.isalnum() and not c.isspace())
    special_ratio = special_count / max(len(text), 1)
    if special_ratio > 0.25:
        return False, f" chars ({special_ratio:.1%} > 25%)"
    
    # rule 4: duplicate — duplicate 
    lines = [l.strip() for l in text.split('\n') if l.strip()]
    if len(lines) > 3:
        unique_ratio = len(set(lines)) / len(lines)
        if unique_ratio < 0.4:
            return False, f"line repetition too high ({unique_ratio:.1%} < 40%)"
    
    # rule 5: share — CAPS LOCK 
    if len(text) > 50:
        upper_ratio = sum(1 for c in text if c.isupper()) / sum(1 for c in text if c.isalpha())
        if upper_ratio > 0.5:
            return False, f"too many uppercase letters ({upper_ratio:.1%} > 50%)"
    
    return True, "pass"


print(f"{' ':<12s} {' ':>8s} {' '}")
print("-" * 55)
for sample in samples:
    passed, reason = quality_check(sample['text'])
    status = "[ok] keep" if passed else "[x] drop"
    print(f"{sample['name']:<12s} {status:>8s}  {reason}")

print()
print(" 20-50 rule.")
print(" about 60-80% Common Crawl .")

=== quality rule ===

                       
-------------------------------------------------------
 document    [ok] keep  pass
             [ok] keep  pass
             [ok] keep  pass
too short    [x] drop  too short (2 words < 5)
random chars [x] drop   chars (29.2% > 25%)
duplicate    [x] drop  line repetition too high (6.5% < 40%)

 20-50 rule.
 about 60-80% Common Crawl .


#### 2.2 Model-based filtering: ask a "language teacher" to score it

In addition to hand-written rules, you can use a small pretrained language model (e.g. KenLM) to score documents.

It is like reading comprehension: a teacher can quickly tell whether a passage is human-written or random noise.

Use perplexity (PPL):

```
PPL very low (< 10): too trivial / repetitive ("a a a a a") -> drop
PPL normal (10-1000): looks human-written -> keep
PPL very high (> 1000): likely garbled -> drop
```

Some systems train a classifier: Wikipedia articles as positive, random Common Crawl as negative.

Key insight: Wikipedia is textbook-grade text; it is a good measuring stick.


---

### 3. Deduplication: don't teach the same sentence 100 times

#### 3.1 Why dedup matters

The web has enormous duplication:
- the same news syndicated across many sites
- the same code snippet copied into many blogs
- placeholder text ("Lorem ipsum")
- cookie banners ("This website uses cookies...")

Without dedup:
1. you waste compute learning repeats
2. the model memorizes instead of learning robust patterns
3. dataset statistics become skewed (some text gets 100x weight)

#### 3.2 Two layers: exact + near-duplicate


In [5]:
# === Exact dedup ===
print("=== Layer 1：Exact dedup (Exact Dedup) ===")
print()

#  5 docsdocument, 2 docs duplicate 
docs = [
    "Machine learning is a subset of artificial intelligence.",
    "Deep learning uses neural networks with many layers.",
    "Machine learning is a subset of artificial intelligence.",  #  doc 1 
    "Natural language processing deals with text data.",
    "Deep learning uses neural networks with many layers.",  #  doc 2 
]

print(f"total {len(docs)} docs")
print()

#  dedup
seen = set()
unique_docs = []

for i, doc in enumerate(docs):
    # SHA256 ： document fingerprint
    fingerprint = hashlib.sha256(doc.encode()).hexdigest()[:16]  #  16 
    is_new = fingerprint not in seen
    
    print(f"doc {i+1}: hash={fingerprint}  {'[ok] keep' if is_new else '[x] duplicate,drop'}")
    
    if is_new:
        seen.add(fingerprint)
        unique_docs.append(doc)

print()
print(f" after: {len(unique_docs)} ({len(docs) - len(unique_docs)} removed)")
print()
print("Exact dedup Removeabout 5-15% Common Crawl ")
print(" —— duplicate 「 」 「 」")

=== Layer 1：Exact dedup (Exact Dedup) ===

total 5 docs

doc 1: hash=a5c7f6a65fc1e1e9  [ok] keep
doc 2: hash=b85a4f441b9eb02a  [ok] keep
doc 3: hash=a5c7f6a65fc1e1e9  [x] duplicate,drop
doc 4: hash=90b6942d04cd6a5c  [ok] keep
doc 5: hash=b85a4f441b9eb02a  [x] duplicate,drop

 after: 3 (2 removed)

Exact dedup Removeabout 5-15% Common Crawl 
 —— duplicate 「 」 「 」


In [6]:
# === MinHash approxdedup： ===
print("=== ：MinHash approxdedup ===")
print()
print(" ：100 docsdocument, ？")
print(" = 100 ² = ")
print()
print("MinHash idea： docsdocument 「fingerprint」,fingerprint -> document ")
print()

# === MinHash ===
print("=== MinHash ===")
print()

#  docsdocument
doc_A = "the cat sat on the mat and looked at the dog"
doc_B = "the cat sat on the mat and watched the dog"  #  A 
doc_C = "quantum mechanics describes behavior of subatomic particles"  #  

print(f"doc A: {doc_A}")
print(f"doc B: {doc_B}")
print(f"doc C: {doc_C}")
print()

# Step 1: docsdocument n-gram ( 3 )
def get_ngrams(text, n=3):
    words = text.lower().split()
    return set(' '.join(words[i:i+n]) for i in range(len(words) - n + 1))

A_ngrams = get_ngrams(doc_A, 3)
B_ngrams = get_ngrams(doc_B, 3)
C_ngrams = get_ngrams(doc_C, 3)

print(f"doc A n-grams ({len(A_ngrams)} ): {A_ngrams}")
print()
print(f"doc B n-grams ({len(B_ngrams)} ): {B_ngrams}")
print()

# Step 2: Jaccard similarity ( / )
def jaccard(s1, s2):
    inter = len(s1 & s2)
    union = len(s1 | s2)
    return inter / union if union > 0 else 0

j_AB = jaccard(A_ngrams, B_ngrams)
j_AC = jaccard(A_ngrams, C_ngrams)

print(f"A ∩ B intersection size: {len(A_ngrams & B_ngrams)}")
print(f"A ∪ B union size: {len(A_ngrams | B_ngrams)}")
print(f"Jaccard(A, B) = {len(A_ngrams & B_ngrams)}/{len(A_ngrams | B_ngrams)} = {j_AB:.2%}")
print()
print(f"Jaccard(A, C) = {j_AC:.2%}")
print()

# Step 3: MinHash fingerprint( —— )
print("=== MinHash ===")
print()

#  ： n-gram , docsdocument K 
def minhash_signature(ngrams, num_hashes=4):
    """ n-gram MinHash MinHash , """
    sig = []
    for i in range(num_hashes):
        #  n-gram seed=i , 
        min_val = float('inf')
        for ng in ngrams:
            h = hash(ng + str(i)) % 100000
            min_val = min(min_val, h)
        sig.append(min_val) if min_val != float('inf') else sig.append(0)
    return sig

sig_A = minhash_signature(A_ngrams)
sig_B = minhash_signature(B_ngrams)
sig_C = minhash_signature(C_ngrams)

print(f"A MinHash signature: {sig_A}")
print(f"B MinHash signature: {sig_B}")
print(f"C MinHash signature: {sig_C}")
print()

# MinHash similarity = 
def minhash_sim(s1, s2):
    matches = sum(1 for a, b in zip(s1, s2) if a == b)
    return matches / len(s1)

mh_AB = minhash_sim(sig_A, sig_B)
mh_AC = minhash_sim(sig_A, sig_C)

print(f"MinHash approx similarity A-B: {mh_AB:.2%} (exact Jaccard: {j_AB:.2%})")
print(f"MinHash approx similarity A-C: {mh_AC:.2%} (exact Jaccard: {j_AC:.2%})")
print()
print("-> MinHash 「 n-gram」 「 4 」")
print("-> ！similarity > ( 80%)-> keep docs")

=== ：MinHash approxdedup ===

 ：100 docsdocument, ？
 = 100 ² = 

MinHash idea： docsdocument 「fingerprint」,fingerprint -> document 

=== MinHash ===

doc A: the cat sat on the mat and looked at the dog
doc B: the cat sat on the mat and watched the dog
doc C: quantum mechanics describes behavior of subatomic particles

doc A n-grams (9 ): {'sat on the', 'on the mat', 'the mat and', 'cat sat on', 'the cat sat', 'at the dog', 'looked at the', 'and looked at', 'mat and looked'}

doc B n-grams (8 ): {'sat on the', 'watched the dog', 'on the mat', 'the mat and', 'and watched the', 'cat sat on', 'the cat sat', 'mat and watched'}

A ∩ B intersection size: 5
A ∪ B union size: 12
Jaccard(A, B) = 5/12 = 41.67%

Jaccard(A, C) = 0.00%

=== MinHash ===

A MinHash signature: [7399, 398, 15128, 584]
B MinHash signature: [9556, 15051, 14853, 584]
C MinHash signature: [53193, 8166, 6632, 25473]

MinHash approx similarity A-B: 25.00% (exact Jaccard: 41.67%)
MinHash approx similarity A-C: 0.00% (exact Jacc

---

### 4. Data mixing: how do we combine sources?

Suppose you have multiple sources:

| Source | Amount | Quality | Role |
|:---|:---|:---|:---|
| Common Crawl (filtered) | 10T tokens | medium | coverage + diversity |
| Wikipedia | 0.1T tokens | very high | factual accuracy |
| Books | 0.5T tokens | high | long-form coherence |
| Code (GitHub) | 1T tokens | medium | logic/programming |
| Academic papers | 0.05T tokens | very high | scientific reasoning |

#### 4.1 Mixing is not "dump everything together"

Strategy: **repeat high-quality data more epochs; repeat low-quality data fewer epochs.**


In [7]:
# === ===
print("=== ： ===")
print()

sources = [
    ("Common Crawl ( )", 10000, 0.6, 1),
    ("Wikipedia",               100, 0.95, 4),
    ("Books",                   500, 0.85, 2),
    ("Code (GitHub)",          1000, 0.75, 2),
    ("ArXiv Papers",             50, 0.9,  4),
    ("News",                    300, 0.7,  1),
]

print(f"{'source':<25s} {'Raw ':>10s} {'quality':>6s} {'Epoch':>6s} {'effective':>10s} {'share':>8s}")
print("-" * 72)

total_effective = 0
results = []
for name, size, quality, epochs in sources:
    effective = size * epochs
    total_effective += effective
    results.append((name, size, quality, epochs, effective))

for name, size, quality, epochs, effective in results:
    ratio = effective / total_effective * 100
    print(f"{name:<25s} {size:>6.0f}B   {quality:>5.0%}  {epochs:>4d}x  {effective:>8.0f}B   {ratio:>6.1f}%")

print()
print(f"total effective data: {total_effective:.0f}B tokens")
print()
print("Keydecisions:")
print(" • Wikipedia 100B, epoch=4 -> 400B ( quality )")
print(" • Common Crawl 10T, epoch=1 -> ( )")
print(" • ArXiv quality ,epoch=4 -> ")
print()
print(" : epoch ≠ document 4 ")
print(" 4 shuffle -> ")

=== ： ===

source                          Raw  quality  Epoch  effective    share
------------------------------------------------------------------------
Common Crawl ( )           10000B     60%     1x     10000B     71.9%
Wikipedia                    100B     95%     4x       400B      2.9%
Books                        500B     85%     2x      1000B      7.2%
Code (GitHub)               1000B     75%     2x      2000B     14.4%
ArXiv Papers                  50B     90%     4x       200B      1.4%
News                         300B     70%     1x       300B      2.2%

total effective data: 13900B tokens

Keydecisions:
 • Wikipedia 100B, epoch=4 -> 400B ( quality )
 • Common Crawl 10T, epoch=1 -> ( )
 • ArXiv quality ,epoch=4 -> 

 : epoch ≠ document 4 
 4 shuffle -> 


---

### 5. Full pipeline demo: prepare data for a 1B model

Now we connect everything and simulate an end-to-end data engineering pipeline.


In [8]:
print("=== Practical: 1B LLM data pipeline ===")
print()

steps = [
    ("Step 1: ", [
        "Chinchilla : N = 1B -> D ≈ 20B tokens",
        " : N = 1B -> D ≈ 100B tokens",
        "Choose: 50B tokens (balanced,good cost/perf)",
    ]),
    ("Step 2: Common Crawl", [
        "download recent 2-3 months dump (about 20TB WARC)",
        "tools: cc_downloader, HuggingFace datasets",
    ]),
    ("Step 3: Text extraction + ", [
        "WARC -> HTML -> (trafilatura / resiliparse)",
        " (fastText): keep ",
        " : ~2TB (about 400B tokens)",
    ]),
    ("Step 4: quality ", [
        " rule: / / / duplicate",
        "KenLM PPL : 10 < PPL < 1000",
        " : ~200GB (about 40B tokens) -> 10%",
    ]),
    ("Step 5: dedup", [
        "Exact dedup: SHA256 -> Remove ~10%",
        "MinHash approxdedup: similarity > 80% keep docs -> Remove ~20%",
        " : ~140GB (about 28B tokens)",
    ]),
    ("Step 6: source", [
        "Wikipedia (2x epoch): 4B tokens",
        "Books (2x epoch): 6B tokens",
        "Code GitHub (2x epoch): 10B tokens",
        " : 2B tokens",
        "Total: 28B + 22B = 50B tokens",
    ]),
    ("Step 7: Tokenize + ", [
        "Use a BPE tokenizer to convert text into token IDs",
        " , 2048/4096 chunk",
        "insert at doc boundary <EOS> token",
        "Shuffle + batch -> ！",
    ]),
]

for title, details in steps:
    print(title)
    for d in details:
        print(f"  {d}")
    print()

print("Compression summary:")
print(" 20TB WARC -> 2TB -> 200GB -> 140GB After dedup")
print(" Raw ~0.7%")

=== Practical: 1B LLM data pipeline ===

Step 1: 
  Chinchilla : N = 1B -> D ≈ 20B tokens
   : N = 1B -> D ≈ 100B tokens
  Choose: 50B tokens (balanced,good cost/perf)

Step 2: Common Crawl
  download recent 2-3 months dump (about 20TB WARC)
  tools: cc_downloader, HuggingFace datasets

Step 3: Text extraction + 
  WARC -> HTML -> (trafilatura / resiliparse)
   (fastText): keep 
   : ~2TB (about 400B tokens)

Step 4: quality 
   rule: / / / duplicate
  KenLM PPL : 10 < PPL < 1000
   : ~200GB (about 40B tokens) -> 10%

Step 5: dedup
  Exact dedup: SHA256 -> Remove ~10%
  MinHash approxdedup: similarity > 80% keep docs -> Remove ~20%
   : ~140GB (about 28B tokens)

Step 6: source
  Wikipedia (2x epoch): 4B tokens
  Books (2x epoch): 6B tokens
  Code GitHub (2x epoch): 10B tokens
   : 2B tokens
  Total: 28B + 22B = 50B tokens

Step 7: Tokenize + 
  Use a BPE tokenizer to convert text into token IDs
   , 2048/4096 chunk
  insert at doc boundary <EOS> token
  Shuffle + batch -> ！

Compressi

### 6. Data quality > data quantity (evidence)

This has been validated repeatedly:

```
T5 (2019):
  750GB cleaned data > 6TB uncleaned data
  -> 1/8 the size, better results

Textbooks Are All You Need (2023):
  a 1.3B model trained on high-quality textbooks can beat larger models on code
  -> quality can compensate for parameter count

LLaMA 2 -> LLaMA 3 (2023-2024):
  architecture changed little; data quality improved a lot
  from ~2T to ~15T tokens while raising quality
  -> better filtering, better dedup, better mixing
```

Intuition: would you rather read 100 great books or 10,000 spam emails? The model feels the same.


### 7. After tokenization: what does the training stream look like?

The last step is tokenization. Then the data becomes a token stream.

```
Doc A: "Machine learning is great."
  -> tokens -> [42, 567, 18, 891, 15]

Doc B: "Deep learning is also great."
  -> tokens -> [123, 567, 18, 456, 891, 15]

Concatenate into one long "token noodle":
[42, 567, 18, 891, 15, <EOS>, 123, 567, 18, 456, 891, 15, <EOS>, ...]

Cut into fixed-length training chunks (seq_len=8):
  chunk 1: [42, 567, 18, 891, 15, <EOS>, 123, 567]
  chunk 2: [18, 456, 891, 15, <EOS>, ...]

Note: chunk boundaries can cut across documents.
A common mitigation is inserting `<EOS>` so the model knows a new document starts.
```


---

## Data Engineering Checklist

1. [ok] Sources: Common Crawl + Wikipedia + Books + Code + Papers
2. [ok] Five-step pipeline: extract -> lang filter -> quality filter -> dedup -> mix
3. [ok] HTML -> Text: remove script/style + strip tags + normalize whitespace
4. [ok] Quality filter: heuristics (length/word/symbols) + PPL scoring
5. [ok] Exact dedup: SHA256, keep only one identical copy
6. [ok] MinHash: n-gram -> hashes -> keep smallest K -> similar signature implies similar doc
7. [ok] Mixing: repeat high-quality sources more epochs
8. [ok] After tokenization: concatenate token stream -> chunk -> train

One-sentence summary: data engineering is not glamorous, but it is the moat. Architectures can be copied; data pipelines are proprietary. You may start with 20TB raw crawl and end with <1% usable clean text.
